In [1]:
import shutil
import pandas as pd
import numpy as np
import os
# torch.manual_seed(1234)
from evaluate_metrics import compute_eer
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
def compute_error_at_threshold(protocol_path, score_path, threshold):
    """고정 threshold에서 EER(=(FAR+FRR)/2) 계산"""
    eval_df = pd.read_csv(protocol_path, sep=" ", header=None, names=["utt", "subset", "label"])
    pred_df = pd.read_csv(score_path, sep=" ", header=None, names=["utt", "spoof", "bonafide"])
    merged = pd.merge(eval_df, pred_df, on="utt")

    bonafide_scores = np.array(merged[merged['label'] == 'bonafide']['bonafide'])
    spoof_scores = np.array(merged[merged['label'] == 'spoof']['bonafide'])

    frr = np.mean(bonafide_scores < threshold)
    far = np.mean(spoof_scores >= threshold)
    eer_fixed = (far + frr) / 2

    print(f"EER(fixed): {eer_fixed*100:.4f}% | FAR: {far*100:.4f}% | FRR: {frr*100:.4f}% | threshold: {threshold:.4f}")
    return eer_fixed, far, frr

In [7]:
import os

# protocol 정리
eval_df = pd.read_csv(
    "/home/woongjae/ADD_LAB/projects/Graduate-Project/02_LoRA_Training/protocols/asv19_eval.txt",
    sep=" ", header=None
)
eval_df.columns = ["utt", 'label','label2']

# 🔥 핵심: 경로 제거 + 확장자 제거
eval_df["utt"] = eval_df["utt"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])


# score
pred_df = pd.read_csv(
    "/home/woongjae/ADD_LAB/projects/Graduate-Project/03_Experiment_Pipeline/experiments/aasist_pretrained_eval/scores_ASV19.txt",
    sep=" ", header=None
)
pred_df.columns = ["utt", "spoof", "bonafide"]

print(eval_df["utt"].head())
print(pred_df["utt"].head())
# merge
res_df = pd.merge(eval_df, pred_df, on='utt')

print("merged len:", len(res_df))  # 👉 이거 꼭 확인


# EER 계산
spoof_scores = res_df[res_df['label'] == 'spoof']['bonafide']
bonafide_scores = res_df[res_df['label'] == 'bonafide']['bonafide']

eer, threshold = compute_eer(bonafide_scores, spoof_scores)

print("EER: {:.4f}%, threshold: {:.4f}".format(eer*100, threshold))

0    LA_E_5617677_LA_E_5617677__auto_tune
1    LA_E_9295475_LA_E_9295475__auto_tune
2    LA_E_2328665_LA_E_2328665__auto_tune
3    LA_E_3341289_LA_E_3341289__auto_tune
4    LA_E_8359549_LA_E_8359549__auto_tune
Name: utt, dtype: object
0    LA_E_5617677_LA_E_5617677__auto_tune
1    LA_E_9295475_LA_E_9295475__auto_tune
2    LA_E_2328665_LA_E_2328665__auto_tune
3    LA_E_3341289_LA_E_3341289__auto_tune
4    LA_E_8359549_LA_E_8359549__auto_tune
Name: utt, dtype: object
merged len: 71236
EER: 82.1481%, threshold: 4.5376


In [19]:
eval_df = pd.read_csv("/home/woongjae/ADD_LAB/Multi_Layer_Fusion/protocols/protocol_itw.txt",sep=" ", header=None)
eval_df.columns = ["utt","subset", 'label']
pred_df = pd.read_csv("/home/woongjae/ADD_LAB/Multi_Layer_Fusion/results/eval_itw.txt", sep=" ", header=None)
pred_df.columns = ["utt", "spoof", "bonafide"]

res_df = pd.merge(eval_df, pred_df, on='utt')

spoof_scores = res_df[res_df['label'] == 'spoof']['bonafide']
bonafide_scores = res_df[res_df['label'] == 'bonafide']['bonafide']

eer, threshold = compute_eer(bonafide_scores, spoof_scores)
print("EER: {:.4f}%, threshold: {:.4f}".format(eer*100, threshold))

EER: 8.1593%, threshold: -3.3488


In [17]:
eval_df = pd.read_csv("/home/woongjae/ADD_LAB/Multi_Layer_Fusion/protocols/protocol_la21.txt",sep=" ", header=None)
eval_df.columns = ["utt","subset", 'label']
pred_df = pd.read_csv("/home/woongjae/ADD_LAB/Multi_Layer_Fusion/results/eval_la21.txt", sep=" ", header=None)
pred_df.columns = ["utt", "spoof", "bonafide"]

res_df = pd.merge(eval_df, pred_df, on='utt')

spoof_scores = res_df[res_df['label'] == 'spoof']['bonafide']
bonafide_scores = res_df[res_df['label'] == 'bonafide']['bonafide']

eer, threshold = compute_eer(bonafide_scores, spoof_scores)
print("EER: {:.4f}%, threshold: {:.4f}".format(eer*100, threshold))

EER: 10.5968%, threshold: 0.8045
